<a href="https://colab.research.google.com/github/pranshulab/Data_science_journey/blob/main/rag_using_langChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q langchain-huggingface sentence-transformers

In [2]:
%pip install -U langchain langchain-core langchain-community langchain-huggingface sentence-transformers faiss-cpu

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
%pip install -U youtube-transcript-api

/tmp/ipykernel_6149/77093084.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

In [61]:
video_id = "0BQpXAEPXS8"  # Only the video ID

try:
    # Create the API instance
    ytt_api = YouTubeTranscriptApi()

    # Fetch the English transcript
    transcript = ytt_api.fetch(video_id, languages=["en"])

    # Convert transcript snippets into plain text
    transcript_text = " ".join(
        snippet.text for snippet in transcript
    )

    print(transcript_text)

except TranscriptsDisabled:
    print("No captions are available for this video.")

except NoTranscriptFound:
    print("No English transcript was found for this video.")

except Exception as e:
    print(f"An error occurred: {type(e).__name__}: {e}")

Friends, welcome to a new video from the most famous city. One of the busiest, one of the biggest and one of the most densely populated cities in the world, New York. So friends, in today's video, we are going to travel from New York City's bright and bustling life to one of the most remote and also one of the least densely populated places on planet Earth. Yes, I am talking about the largest state of the USA, Alaska. Friends, Alaska is such a big state that if we take the entire population of the world and put it inside Alaska, it can easily fit there. Now, Alaska is a separate state of the USA, but it lies between Russia and Canada. And even here, Alaska does not share a land border of up to one kilometer with US state. In fact, Alaska is only less than four kilometers away from Russia. In the morning, all New Yorkers are in action. Every person looks in a hurry. In Manhattan, New York, everyone’s rushing to work early. Right now, I am standing on 33rd Street, 8th Avenue. And from he

In [62]:
transcript_text

"Friends, welcome to a new video from the most famous city. One of the busiest, one of the biggest and one of the most densely populated cities in the world, New York. So friends, in today's video, we are going to travel from New York City's bright and bustling life to one of the most remote and also one of the least densely populated places on planet Earth. Yes, I am talking about the largest state of the USA, Alaska. Friends, Alaska is such a big state that if we take the entire population of the world and put it inside Alaska, it can easily fit there. Now, Alaska is a separate state of the USA, but it lies between Russia and Canada. And even here, Alaska does not share a land border of up to one kilometer with US state. In fact, Alaska is only less than four kilometers away from Russia. In the morning, all New Yorkers are in action. Every person looks in a hurry. In Manhattan, New York, everyone’s rushing to work early. Right now, I am standing on 33rd Street, 8th Avenue. And from h

In [63]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_text])

In [64]:
len(chunks)

45

In [65]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [66]:
vector_store.index_to_docstore_id

{0: 'f5851846-3472-4924-866c-41306cfa0ca5',
 1: '10785014-3e24-457e-a9f6-6f833594b576',
 2: '1dc33132-8dc5-4739-9951-30c5659766a4',
 3: 'aa8af671-1ff4-4ac5-805a-1a4ab150ab66',
 4: '1326f5f3-3880-4fe3-91fc-93a8bc5e88b3',
 5: '8e796096-2b42-494d-98cc-2e68b2edf064',
 6: '84ca357e-d986-420d-89b8-95cda40df513',
 7: 'e01c02f6-5027-4e59-8f37-1efa12ad6a46',
 8: 'debe5a90-2214-4d62-b866-f577ce26882c',
 9: '9c710e92-5187-47e7-a610-7775dfd8530e',
 10: '5280c90b-f089-4d9f-82ae-2ccb824e87ba',
 11: '6dce8f5f-dc17-4f68-963a-9981a19ea578',
 12: '19aa3b1b-5455-45cd-9d5e-820be840d007',
 13: '02ddc850-afe4-4bc6-a539-101bbfe5bb25',
 14: '49c82847-8a74-43c6-97c7-0dd55c51885f',
 15: '52373f87-8376-4195-b238-c0042f32a607',
 16: '94e97d92-f3b0-401b-94f2-d08310687da2',
 17: '8fc57057-909b-4d3f-bc3c-2ed968121f74',
 18: 'd9c6aff8-58e1-411b-ac82-539c17cd9cd6',
 19: '1196b71b-e9d3-465c-a007-19c80c5fa647',
 20: '8c195255-3634-4e67-a2ee-ff057815b3ef',
 21: '70236263-d2a1-42e1-bfd5-a0dd8747e1bd',
 22: 'bc9de7e2-4281-

In [68]:
vector_store.get_by_ids(['43ea2648-d772-47b4-ab39-0b5000159386'])

[Document(id='43ea2648-d772-47b4-ab39-0b5000159386', metadata={}, page_content='common bathrooms, etc. But there are also many free campsites. Inside the city and outside the city. So one of them is this free campsite in Anchorage. Which is the only one for now. And because the season is over now. Winter has started now. From September and October. So all the free sites here will be closed, right? If there were 10 here, then 6 were closed. Okay. Not the free ones. Okay, the camping grounds are closed. Yes, the camping grounds, the paid ones. Okay, the paid ones. All are closed. 3-4 of them are left here. And their price is double. Okay. And they also have a rule. They say, park it at night. But go in the morning. Okay. Oh, the storm has come. Yes, the storm has come. I have just parked here. A few days ago, where my van was parked, there was another van with it. I was making a video there. Yes. That big tree was falling as I was filming for the video. I ran to that girl. I said, the tr

**Step 2 - Retrieval**

In [69]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [70]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b2bad7eb9d0>, search_kwargs={'k': 1})

In [71]:
retriever.invoke('What is RAG')

[Document(id='dd23856f-c413-47f6-867f-718aa0b78019', metadata={}, page_content="city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Okay. I told him to take it. Deepanshu. Our Nomadic Indian brother. You must have understood. But brother, look a

**Step 3 - Augmentation**

In [72]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [87]:
question          = "is the topic of homeless is discused in this video if yes what was that"
retrieved_docs    = retriever.invoke(question)

In [88]:
retrieved_docs

[Document(id='dd23856f-c413-47f6-867f-718aa0b78019', metadata={}, page_content="city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Okay. I told him to take it. Deepanshu. Our Nomadic Indian brother. You must have understood. But brother, look a

In [89]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Okay. I told him to take it. Deepanshu. Our Nomadic Indian brother. You must have understood. But brother, look at this. We are in a state of the US. And right now, I am closer to Tokyo. Which

In [90]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [91]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Ok

**Step 4 - Generation**

In [78]:
%pip install -q -U transformers accelerate langchain-huggingface

In [92]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

In [93]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [94]:
llm = HuggingFacePipeline(pipeline=pipe)

In [95]:
answer = llm.invoke(final_prompt)
print(answer)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Okay. I told him to take it. De

**Building a Chain**

In [96]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [97]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [98]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [99]:
parallel_chain.invoke('honeless in Alaska')

{'context': "city of Alaska. This is also very dangerous. This is one of the highest crime-rated cities in the US. Brother, have you heard here? I know. You know why it was made. What happened is that. In the United States, like in New York, LA, there were a lot of homeless people. Yes, yes. So what did they do? They caught the homeless. Okay. They started leaving them here. Okay. And they knew. They can't go anywhere while coming from Canada. Yes. They will not go by swimming through the sea. Yes. They have sent so many homeless people here. Okay. They have sent more than  10,000 homeless people here. Okay. In Anchorage. And even today. I was washing clothes. Homeless people were roaming there. I have given one jacket. Okay. They said they were feeling cold. I gave him the jacket. It was Deepanshu's jacket. Okay. I told him to take it. Deepanshu. Our Nomadic Indian brother. You must have understood. But brother, look at this. We are in a state of the US. And right now, I am closer to 

In [100]:
parser = StrOutputParser()

In [101]:
main_chain = parallel_chain | prompt | llm | parser

In [102]:
main_chain.invoke('Can you summarize the video')

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      making a video there. Yes. That big tree was falling as I was filming for the video. I ran to that girl. I said, the tree fell near your van. Friends, look outside. The storm is coming. The wind is blowing very fast. Oh brother, I don't know how much you are able to hear or not. But brother, this is a dangerous place, Alaska. So friends, this was our journey today. It's been 24 hours since I left New York. First, I took a train from Long Island to Manhattan. Then I took a train from Manhattan to Philadelphia. Then I took a train from Philadelphia to Philadelphia Airport. Then I took a flight from Philadelphia Airport to Minneapolis Airport. After that, you saw the flight from Minneapolis Airport. The entire runway situation. Finally, the flight flew and I landed. I landed at Anchorage Airport in Alaska. From there, Sunty